# PySpark

![Logo](https://github.com/pnavaro/big-data/blob/master/notebooks/images/apache_spark_logo.png?raw=1)

In [174]:
pip install pyspark

- [Apache Spark](https://spark.apache.org) was first released in 2014.
- It was originally developed by [Matei Zaharia](http://people.csail.mit.edu/matei) as a class project, and later a PhD dissertation, at University of California, Berkeley.
- Spark is written in [Scala](https://www.scala-lang.org).
- All images come from [Databricks](https://databricks.com/product/getting-started-guide).

- Apache Spark is a fast and general-purpose cluster computing system.
- It provides high-level APIs in Java, Scala, Python and R, and an optimized engine that supports general execution graphs.
- Spark can manage "big data" collections with a small set of high-level primitives like `map`, `filter`, `groupby`, and `join`.  With these common patterns we can often handle computations that are more complex than map, but are still structured.
- It also supports a rich set of higher-level tools including [Spark SQL](https://spark.apache.org/docs/latest/sql-programming-guide.html) for SQL and structured data processing, [MLlib](https://spark.apache.org/docs/latest/ml-guide.html) for machine learning, [GraphX](https://spark.apache.org/docs/latest/graphx-programming-guide.html) for graph processing, and Spark Streaming.

## Resilient distributed datasets

- The fundamental abstraction of Apache Spark is a read-only, parallel, distributed, fault-tolerent collection called a resilient distributed datasets (RDD).
- RDDs behave a bit like Python collections (e.g. lists).
- When working with Apache Spark we iteratively apply functions to every item of these collections in parallel to produce *new* RDDs.
- The data is distributed across nodes in a cluster of computers.
- Functions implemented in Spark can work in parallel across elements of the collection.
- The  Spark framework allocates data and processing to different nodes, without any intervention from the programmer.
- RDDs automatically rebuilt on machine failure.

## Lifecycle of a Spark Program

1. Create some input RDDs from external data or parallelize a collection in your driver program.
2. Lazily transform them to define new RDDs using transformations like `filter()` or `map()`
3. Ask Spark to cache() any intermediate RDDs that will need to be reused.
4. Launch actions such as count() and collect() to kick off a parallel computation, which is then optimized and executed by Spark.

## Operations on Distributed Data

- Two types of operations: **transformations** and **actions**
- Transformations are *lazy* (not computed immediately)
- Transformations are executed when an action is run

## [Transformations](https://spark.apache.org/docs/latest/rdd-programming-guide.html#transformations) (lazy)

```
map() flatMap()
filter()
mapPartitions() mapPartitionsWithIndex()
sample()
union() intersection() distinct()
groupBy() groupByKey()
reduceBy() reduceByKey()
sortBy() sortByKey()
join()
cogroup()
cartesian()
pipe()
coalesce()
repartition()
partitionBy()
...
```

## [Actions](https://spark.apache.org/docs/latest/rdd-programming-guide.html#actions)

```
reduce()
collect()
count()
first()
take()
takeSample()
saveToCassandra()
takeOrdered()
saveAsTextFile()
saveAsSequenceFile()
saveAsObjectFile()
countByKey()
foreach()
```

## Python API

PySpark uses Py4J that enables Python programs to dynamically access Java objects.

![PySpark Internals](https://github.com/pnavaro/big-data/blob/master/notebooks/images/YlI8AqEl.png?raw=1)

## The `SparkContext` class

- When working with Apache Spark we invoke methods on an object which is an instance of the `pyspark.SparkContext` context.

- Typically, an instance of this object will be created automatically for you and assigned to the variable `sc`.

- The `parallelize` method in `SparkContext` can be used to turn any ordinary Python collection into an RDD;
    - normally we would create an RDD from a large file or an HBase table.

## First example

PySpark isn't on sys.path by default, but that doesn't mean it can't be used as a regular library. You can address this by either symlinking pyspark into your site-packages, or adding pyspark to sys.path at runtime. [findspark](https://github.com/minrk/findspark) does the latter.

We have a spark context sc to use with a tiny local spark cluster with 4 nodes (will work just fine on a multicore machine).

In [175]:
import os, sys
sys.executable

'/usr/bin/python3'

In [176]:
#os.environ["SPARK_HOME"] = "/opt/spark-3.0.1-bin-hadoop2.7"
os.environ["PYSPARK_PYTHON"] = sys.executable

In [177]:
import pyspark

# Stop any existing SparkContext before initializing a new one
if 'sc' in globals() and sc:
    sc.stop()

# Use getOrCreate to reuse an existing SparkContext or create a new one
sc = pyspark.SparkContext.getOrCreate(conf=pyspark.SparkConf().setMaster("local[*]").setAppName("FirstExample"))
sc.setLogLevel("ERROR")
print(sc)

<SparkContext master=local[*] appName=FirstExample>


In [178]:
print('Files in /content/ directory:')
!ls -F /content/

Files in /content/ directory:
data/  sample_data/  sample.txt


In [179]:
print(sc) # it is like a Pool Processor executor

<SparkContext master=local[*] appName=FirstExample>


## Create your first RDD

In [180]:
data = list(range(8))
rdd = sc.parallelize(data) # create collection
rdd

ParallelCollectionRDD[0] at readRDDFromFile at PythonRDD.scala:298

### Exercise

Create a file `sample.txt`with lorem package. Read and load it into a RDD with the `textFile` spark function.

In [181]:
from faker import Faker
fake = Faker()
Faker.seed(0)

with open("sample.txt","w") as f:
    f.write(fake.text(max_nb_chars=1000))

rdd = sc.textFile("sample.txt")

### Collect

Action / To Driver: Return all items in the RDD to the driver in a single list

![](https://github.com/pnavaro/big-data/blob/master/notebooks/images/DUO6ygB.png?raw=1)

Source: https://i.imgur.com/DUO6ygB.png

### Exercise

Collect the text you read before from the `sample.txt`file.

### Map

Transformation / Narrow: Return a new RDD by applying a function to each element of this RDD

![](https://github.com/pnavaro/big-data/blob/master/notebooks/images/PxNJf0U.png?raw=1)

Source: http://i.imgur.com/PxNJf0U.png

In [182]:
rdd = sc.parallelize(list(range(8)))
rdd.map(lambda x: x ** 2).collect() # Square each element

[0, 1, 4, 9, 16, 25, 36, 49]

### Exercise

Replace the lambda function by a function that contains a pause (sleep(1)) and check if the `map` operation is parallelized.

In [183]:
import time

def my_sleep_function(x):
    time.sleep(1)
    return x * 2

# Define the number of partitions
num_partitions = 4 # You can change this value to observe different execution times

# Create a larger RDD to better observe parallelization, using the defined number of partitions
rdd_sleep = sc.parallelize(range(8), numSlices=num_partitions)

print(f"Starting parallel map operation with {num_partitions} partitions...")
start_time = time.time()
results = rdd_sleep.map(my_sleep_function).collect()
end_time = time.time()

print(f"Results: {results}")
print(f"Execution time for parallel map with {num_partitions} partitions: {end_time - start_time:.2f} seconds")

Starting parallel map operation with 4 partitions...
Results: [0, 2, 4, 6, 8, 10, 12, 14]
Execution time for parallel map with 4 partitions: 4.17 seconds


In [184]:
print("Data distribution across partitions using glom():")
# glom() transforms an RDD of items into an RDD of lists, where each list contains all items from a partition.
partitions_data = rdd_sleep.glom().collect()

for i, partition in enumerate(partitions_data):
    print(f"Partition {i}: {partition}")

Data distribution across partitions using glom():
Partition 0: [0, 1]
Partition 1: [2, 3]
Partition 2: [4, 5]
Partition 3: [6, 7]


In [185]:
print("Filtering rdd_sleep to keep only even numbers:")
even_numbers_rdd = rdd_sleep.filter(lambda x: x % 2 == 0)
print(f"Even numbers from rdd_sleep: {even_numbers_rdd.collect()}")

Filtering rdd_sleep to keep only even numbers:
Even numbers from rdd_sleep: [0, 2, 4, 6]


### Comparing execution times with different number of partitions

In [186]:
import time

def my_sleep_function(x):
    time.sleep(1)
    return x * 2

# Define a list of partition numbers to test
partition_values = [1, 2, 4, 8] # You can add or modify these values

for num_partitions in partition_values:
    print(f"\n--- Testing with {num_partitions} partitions ---")

    # Re-parallelize the data with the new number of partitions
    rdd_sleep_test = sc.parallelize(range(8), numSlices=num_partitions)

    print(f"Starting parallel map operation with {num_partitions} partitions...")
    start_time = time.time()
    results = rdd_sleep_test.map(my_sleep_function).collect()
    end_time = time.time()

    print(f"Results: {results}")
    print(f"Execution time for parallel map with {num_partitions} partitions: {end_time - start_time:.2f} seconds")

    # Optionally, show distribution for each case
    print("Data distribution across partitions using glom():")
    partitions_data = rdd_sleep_test.glom().collect()
    for i, partition in enumerate(partitions_data):
        print(f"Partition {i}: {partition}")


--- Testing with 1 partitions ---
Starting parallel map operation with 1 partitions...
Results: [0, 2, 4, 6, 8, 10, 12, 14]
Execution time for parallel map with 1 partitions: 8.07 seconds
Data distribution across partitions using glom():
Partition 0: [0, 1, 2, 3, 4, 5, 6, 7]

--- Testing with 2 partitions ---
Starting parallel map operation with 2 partitions...
Results: [0, 2, 4, 6, 8, 10, 12, 14]
Execution time for parallel map with 2 partitions: 4.08 seconds
Data distribution across partitions using glom():
Partition 0: [0, 1, 2, 3]
Partition 1: [4, 5, 6, 7]

--- Testing with 4 partitions ---
Starting parallel map operation with 4 partitions...
Results: [0, 2, 4, 6, 8, 10, 12, 14]
Execution time for parallel map with 4 partitions: 4.20 seconds
Data distribution across partitions using glom():
Partition 0: [0, 1]
Partition 1: [2, 3]
Partition 2: [4, 5]
Partition 3: [6, 7]

--- Testing with 8 partitions ---
Starting parallel map operation with 8 partitions...
Results: [0, 2, 4, 6, 8, 

### Filter

Transformation / Narrow: Return a new RDD containing only the elements that satisfy a predicate

![](https://github.com/pnavaro/big-data/blob/master/notebooks/images/GFyji4U.png?raw=1)
Source: http://i.imgur.com/GFyji4U.png

In [187]:
# Select only the even elements
rdd.filter(lambda x: x % 2 == 0).collect()

[0, 2, 4, 6]

In [188]:
import pyspark

# Stop any existing SparkContext before initializing a new one
if 'sc' in globals() and sc:
    sc.stop()

# Initialize SparkContext with a local master and an app name
sc = pyspark.SparkContext(master="local[*]", appName="SparkContextInitialization")
sc.setLogLevel("ERROR") # Suppress excessive logging
print(sc)

<SparkContext master=local[*] appName=SparkContextInitialization>


### FlatMap

Transformation / Narrow: Return a new RDD by first applying a function to all elements of this RDD, and then flattening the results

![](https://github.com/pnavaro/big-data/blob/master/notebooks/images/TsSUex8.png?raw=1)

In [189]:
rdd = sc.parallelize([1,2,3])
rdd.flatMap(lambda x: (x, x*100, 42)).collect()

[1, 100, 42, 2, 200, 42, 3, 300, 42]

### FlatMap Exercise: Clean `sample.txt`

In [190]:
# Install Faker if not already installed
!pip install faker

In [191]:
# Re-create sample.txt as Faker was not installed previously
from faker import Faker
fake = Faker()
Faker.seed(0)

with open("sample.txt","w") as f:
    f.write(fake.text(max_nb_chars=1000))

print("sample.txt re-created successfully.")

sample.txt re-created successfully.


In [192]:
# Load sample.txt into an RDD
rdd_text = sc.textFile("sample.txt")

# Apply flatMap to clean the text: lowercase, remove dots, and split into words
# We'll also filter out empty strings that might result from splitting
cleaned_words_rdd = rdd_text.flatMap(lambda line: line.lower().replace('.', '').split())

# Display the first 20 cleaned words to verify
print("First 20 cleaned words:")
print(cleaned_words_rdd.take(20))

First 20 cleaned words:
['american', 'whole', 'magazine', 'truth', 'stop', 'whose', 'on', 'traditional', 'measure', 'example', 'sense', 'peace', 'would', 'mouth', 'relate', 'own', 'chair', 'together', 'range', 'line']


### Exercise

Use FlatMap to clean the text from `sample.txt`file. Lower, remove dots and split into words.

### GroupBy

Transformation / Wide: Group the data in the original RDD. Create pairs where the key is the output of a user function, and the value is all items for which the function yields this key.

![](https://github.com/pnavaro/big-data/blob/master/notebooks/images/gdj0Ey8.png?raw=1)

### Example: RDD `join()` operation

In [193]:
# Create the first RDD with key-value pairs
rdd1 = sc.parallelize([
    ("apple", 10),
    ("banana", 5),
    ("orange", 8),
    ("grape", 15)
])

# Create the second RDD with key-value pairs
rdd2 = sc.parallelize([
    ("apple", "red"),
    ("banana", "yellow"),
    ("orange", "orange"),
    ("kiwi", "green"),
    ("apple", "green") # Another entry for 'apple'
])

print("RDD1 elements:", rdd1.collect())
print("RDD2 elements:", rdd2.collect())

RDD1 elements: [('apple', 10), ('banana', 5), ('orange', 8), ('grape', 15)]
RDD2 elements: [('apple', 'red'), ('banana', 'yellow'), ('orange', 'orange'), ('kiwi', 'green'), ('apple', 'green')]


In [194]:
# Perform an inner join on the RDDs
# The result will contain pairs of (key, (value1, value2))
# For keys that appear multiple times in either RDD, all combinations will be returned.
joined_rdd = rdd1.join(rdd2)

print("Joined RDD elements:")
print(joined_rdd.collect())

Joined RDD elements:
[('apple', (10, 'red')), ('apple', (10, 'green')), ('banana', (5, 'yellow')), ('orange', (8, 'orange'))]


Counting key occurrences in `rdd2` using `reduceByKey`

In [195]:
from operator import add

# Re-define rdd1 and rdd2 to ensure they are available
rdd1 = sc.parallelize([
    ("apple", 10),
    ("banana", 5),
    ("orange", 8),
    ("grape", 15)
])
rdd2 = sc.parallelize([
    ("apple", "red"),
    ("banana", "yellow"),
    ("orange", "orange"),
    ("kiwi", "green"),
    ("apple", "green") # Another entry for 'apple'
])

# Transform rdd2 into (key, 1) pairs to count occurrences
rdd2_for_counting = rdd2.map(lambda x: (x[0], 1))

print("RDD2 transformed for counting (key, 1):")
print(rdd2_for_counting.collect())

# Use reduceByKey to sum the counts for each key
key_counts = rdd2_for_counting.reduceByKey(add)

print("\nKey occurrences in rdd2:")
print(key_counts.collect())

RDD2 transformed for counting (key, 1):
[('apple', 1), ('banana', 1), ('orange', 1), ('kiwi', 1), ('apple', 1)]

Key occurrences in rdd2:
[('apple', 2), ('banana', 1), ('kiwi', 1), ('orange', 1)]


As you can see, the `join()` operation combines elements from `rdd1` and `rdd2` where the keys match. If a key appears multiple times in either RDD, it produces all possible combinations of values for that key.

In [196]:
# Perform a left outer join on the RDDs
# This will return all keys from rdd1, and matching values from rdd2.
# If no match in rdd2, it will show None for rdd2's value.
left_joined_rdd = rdd1.leftOuterJoin(rdd2)

print("Left Outer Joined RDD elements:")
print(left_joined_rdd.collect())

Left Outer Joined RDD elements:
[('apple', (10, 'red')), ('apple', (10, 'green')), ('banana', (5, 'yellow')), ('orange', (8, 'orange')), ('grape', (15, None))]


In [197]:
rdd = sc.parallelize(['John', 'Fred', 'Anna', 'James'])
rdd = rdd.groupBy(lambda w: w[0])
[(k, list(v)) for (k, v) in rdd.collect()]

[('J', ['John', 'James']), ('F', ['Fred']), ('A', ['Anna'])]

### GroupByKey

Transformation / Wide: Group the values for each key in the original RDD. Create a new pair where the original key corresponds to this collected group of values.

![](https://github.com/pnavaro/big-data/blob/master/notebooks/images/TlWRGr2.png?raw=1)

In [198]:
rdd = sc.parallelize([('B',5),('B',4),('A',3),('A',2),('A',1)])
rdd = rdd.groupByKey()
[(j[0], list(j[1])) for j in rdd.collect()]

[('A', [3, 2, 1]), ('B', [5, 4])]

### Join

Transformation / Wide: Return a new RDD containing all pairs of elements having the same key in the original RDDs

![](https://github.com/pnavaro/big-data/blob/master/notebooks/images/YXL42Nl.png?raw=1)

In [199]:
x = sc.parallelize([("a", 1), ("b", 2)])
y = sc.parallelize([("a", 3), ("a", 4), ("b", 5)])
x.join(y).collect()

[('b', (2, 5)), ('a', (1, 3)), ('a', (1, 4))]

### Distinct

Transformation / Wide: Return a new RDD containing distinct items from the original RDD (omitting all duplicates)

![](https://github.com/pnavaro/big-data/blob/master/notebooks/images/Vqgy2a4.png?raw=1)

In [200]:
rdd = sc.parallelize([1,2,3,3,4])
rdd.distinct().collect()

[2, 4, 1, 3]

### KeyBy

Transformation / Narrow: Create a Pair RDD, forming one pair for each item in the original RDD. The pair’s key is calculated from the value via a user-supplied function.

![](https://github.com/pnavaro/big-data/blob/master/notebooks/images/nqYhDW5.png?raw=1)

In [201]:
rdd = sc.parallelize(['John', 'Fred', 'Anna', 'James'])
rdd.keyBy(lambda w: w[0]).collect()

[('J', 'John'), ('F', 'Fred'), ('A', 'Anna'), ('J', 'James')]

## Actions

### Map-Reduce operation

Action / To Driver: Aggregate all the elements of the RDD by applying a user function pairwise to elements and partial results, and return a result to the driver

![](https://github.com/pnavaro/big-data/blob/master/notebooks/images/R72uzwX.png?raw=1)

In [202]:
from operator import add
rdd = sc.parallelize(list(range(8)))
rdd.map(lambda x: x ** 2).reduce(add) # reduce is an action!

140

### Max, Min, Sum, Mean, Variance, Stdev

Action / To Driver: Compute the respective function (maximum value, minimum value, sum, mean, variance, or standard deviation) from a numeric RDD

![](https://github.com/pnavaro/big-data/blob/master/notebooks/images/HUCtib1.png?raw=1)

### CountByKey

Action / To Driver: Return a map of keys and counts of their occurrences in the RDD

![](https://github.com/pnavaro/big-data/blob/master/notebooks/images/jvQTGv6.png?raw=1)

In [203]:
rdd = sc.parallelize([('J', 'James'), ('F','Fred'),
                    ('A','Anna'), ('J','John')])

rdd.countByKey()

defaultdict(int, {'J': 2, 'F': 1, 'A': 1})

In [204]:
# Stop the local spark cluster
sc.stop()

### Exercise 10.1 Word-count in Apache Spark

- Write the sample text file

- Create the rdd with `SparkContext.textFile method`
- lower, remove dots and split using `rdd.flatMap`
- use `rdd.map` to create the list of key/value pair (word, 1)
- `rdd.reduceByKey` to get all occurences
- `rdd.takeOrdered`to get sorted frequencies of words

All documentation is available [here](https://spark.apache.org/docs/2.1.0/api/python/pyspark.html?highlight=textfile#pyspark.SparkContext) for textFile and [here](https://spark.apache.org/docs/2.1.0/api/python/pyspark.html?highlight=textfile#pyspark.RDD) for RDD.

For a global overview see the Transformations section of the [programming guide](https://spark.apache.org/docs/latest/rdd-programming-guide.html)


## SparkSession

Since SPARK 2.0.0,  SparkSession provides a single point
of entry to interact with Spark functionality and
allows programming Spark with DataFrame and Dataset APIs.

###  $\pi$ computation example

- We can estimate an approximate value for $\pi$ using the following Monte-Carlo method:

1.    Inscribe a circle in a square
2.    Randomly generate points in the square
3.    Determine the number of points in the square that are also in the circle
4.    Let $r$ be the number of points in the circle divided by the number of points in the square, then $\pi \approx 4 r$.
    
- Note that the more points generated, the better the approximation

See [this tutorial](https://computing.llnl.gov/tutorials/parallel_comp/#ExamplesPI).


### Exercise 9.2

Using the same method than the PI computation example, compute the integral
$$
I = \int_0^1 \exp(-x^2) dx
$$
You can check your result with numpy

In [205]:
# numpy evaluates solution using numeric computation.
# It uses discrete values of the function
import numpy as np
x = np.linspace(0,1,1000)
np.trapz(np.exp(-x*x),x)

/tmp/ipykernel_3308/3727401733.py:5: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  np.trapz(np.exp(-x*x),x)


np.float64(0.7468240713763741)

numpy and scipy evaluates solution using numeric computation. It uses discrete values of the function

In [206]:
import numpy as np
from scipy.integrate import quad
quad(lambda x: np.exp(-x*x), 0, 1)
# note: the solution returned is complex

(0.7468241328124271, 8.291413475940725e-15)

### Correlation between daily stock

- Data preparation

In [207]:
import os

# Create the 'data' directory if it doesn't exist
if not os.path.exists('data'):
    os.makedirs('data')
    print("Created directory: data")
else:
    print("Directory 'data' already exists.")


Directory 'data' already exists.


In [208]:
# !!! IMPORTANT: Replace 'YOUR_DOWNLOAD_LINK_HERE' with the actual URL for daily-stock.tgz
# Example: !wget -P data/ https://example.com/daily-stock.tgz
!wget -P data/ YOUR_DOWNLOAD_LINK_HERE


--2026-09-22 19:13:14--  http://your_download_link_here/
Resolving your_download_link_here (your_download_link_here)... failed: Name or service not known.
wget: unable to resolve host address ‘your_download_link_here’


Let's check the contents of the `/content/data/` directory to confirm the `daily-stock.tgz` file has been uploaded.

In [209]:
print('Files in /content/data/ directory:')
!ls -F /content/data/

Files in /content/data/ directory:


Now that we've confirmed the `daily-stock.tgz` file is present, let's extract its contents using the `extract_data` function.

In [211]:
extract_data('daily-stock','data') # this function call will extract json files

Extracting data...


FileNotFoundError: [Errno 2] No such file or directory: 'data/daily-stock.tgz'

In [215]:
print('Files in /content/data/ directory:')
!ls -F /content/data/

Files in /content/data/ directory:


In [ ]:
print('Files in /content/ directory:')
!ls -F /content/

In [ ]:
print('Files in /content/ directory:')
!ls -F /content/

In [ ]:
import os  # library to get directory and file paths
import tarfile # this module makes possible to read and write tar archives

def extract_data(name, where):
    datadir = os.path.join(where,name)
    if not os.path.exists(datadir):
       print("Extracting data...")
       tar_path = os.path.join(where, name+'.tgz')
       with tarfile.open(tar_path, mode='r:gz') as data:
          data.extractall(where)

extract_data('daily-stock','data') # this function call will extract json files

In [212]:
import json
import pandas as pd
import os, glob

here = os.getcwd()
datadir = os.path.join(here,'data','daily-stock')
filenames = sorted(glob.glob(os.path.join(datadir, '*.json')))
filenames

[]

In [213]:
%rm data/daily-stock/*.h5

rm: cannot remove 'data/daily-stock/*.h5': No such file or directory


In [ ]:
from glob import glob
import os, json
import pandas as pd

for fn in filenames:
    with open(fn) as f:
        data = [json.loads(line) for line in f]

    df = pd.DataFrame(data)

    out_filename = fn[:-5] + '.h5'
    df.to_hdf(out_filename, '/data')
    print("Finished : %s" % out_filename.split(os.path.sep)[-1])

filenames = sorted(glob(os.path.join('data', 'daily-stock', '*.h5')))  # data/json/*.json

### Sequential code

In [ ]:
filenames

In [ ]:
with pd.HDFStore('data/daily-stock/aet.h5') as hdf:
    # This prints a list of all group names:
    print(hdf.keys())

In [ ]:
df_test = pd.read_hdf('data/daily-stock/aet.h5')

In [ ]:
%%time

series = []
for fn in filenames:   # Simple map over filenames
    series.append(pd.read_hdf(fn)["close"])

results = []

for a in series:    # Doubly nested loop over the same collection
    for b in series:
        if not (a == b).all():     # Filter out comparisons of the same series
            results.append(a.corr(b))  # Apply function

result = max(results)
result

### Exercise 9.3

Parallelize the code above with Apache Spark.

- Change the filenames because of the Hadoop environment.

In [ ]:
import os, glob

here = os.getcwd()
filenames = sorted(glob.glob(os.path.join(here,'data', 'daily-stock', '*.h5')))
filenames

If it is not started don't forget the PySpark context

Computation time is slower because there is a lot of setup, workers creation, there is a lot of communications the correlation function is too small

### Exercise 9.4 Fasta file example

Use a RDD to calculate the GC content of fasta file nucleotide-sample.txt:

$$\frac{G+C}{A+T+G+C}\times100 \% $$

Create a rdd from fasta file genome.txt in data directory and count 'G' and 'C' then divide by the total number of bases.

### Another example

Compute the most frequent sequence with 5 bases.